# 7주차 ③ 데이터 증강과 혼합정밀도(AMP) — 실습 7~9

**목표**: `transforms v2` 로 증강을 적용해 과적합이 줄어드는 것을 검증 곡선으로 확인하고,
`autocast` + `GradScaler` 로 AMP 를 적용해 **학습 시간과 VRAM 개선을 측정**한다.

```
   과적합 = 데이터가 적다 + 모델이 크다

     해법 ①  데이터를 늘린다      ← 가장 확실. 그런데 대개 못 구한다
     해법 ②  모델을 줄인다        ← 성능도 같이 준다
     해법 ③  드롭아웃·가중치감쇠   ← 6주차에 배움
     해법 ④  있는 데이터를 변형한다  ★ 오늘 — 사실상 ①의 값싼 버전
```

```
   고양이 사진을 좌우로 뒤집으면?     여전히 고양이  ○ 써도 된다
   조금 잘라내면?                     여전히 고양이  ○
   색을 조금 바꾸면?                   여전히 고양이  ○
   상하로 뒤집으면?                    ...CIFAR-10 에서는 부자연스럽다  △
   숫자 6 을 180도 돌리면?             9 가 된다      ✗ 절대 안 됨
```

> **핵심 메시지 ★**: 증강의 원칙은 **"라벨이 바뀌지 않는 변형만"** 입니다.

> **핵심 메시지 ★★ (출제 지점)**: **검증·테스트에는 증강을 적용하지 않습니다.**
> 증강은 **학습을 어렵게 만들어 일반화를 돕는 장치**입니다.
> 평가에까지 적용하면 **매번 다른 이미지를 평가하는 셈**이라 점수를 믿을 수 없습니다.
> 그래서 `train_tf` 와 `test_tf` 를 **따로** 만듭니다.

## 실습 7 — `transforms v2` 증강 4종

In [ ]:
# 셀 1 — 증강 있는 변환 / 없는 변환
import torch, torch.nn as nn, time
from torchvision import datasets
from torchvision.transforms import v2                 # ★ v2 를 쓴다
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

device = "cuda" if torch.cuda.is_available() else "cpu"
print("장치 :", device)
CLASSES = ["비행기","자동차","새","고양이","사슴","개","개구리","말","배","트럭"]
MEAN, STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

# 학습용 : 증강 있음
train_tf = v2.Compose([
    v2.RandomCrop(32, padding=4),        # 4픽셀 덧대고 32로 자르기 → 위치 흔들기
    v2.RandomHorizontalFlip(p=0.5),      # 좌우 뒤집기
    v2.ColorJitter(brightness=0.2, contrast=0.2),
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])
# 평가용 : 증강 없음  ★
test_tf = v2.Compose([
    v2.ToImage(), v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(MEAN, STD),
])

In [ ]:
# 셀 2 — 같은 이미지를 6번 뽑아 본다
raw = datasets.CIFAR10("data", train=True, download=True)
img, label = raw[7]

fig, ax = plt.subplots(1, 7, figsize=(14, 2.4))
ax[0].imshow(img); ax[0].set_title(f"원본\n{CLASSES[label]}"); ax[0].axis("off")
for i in range(1, 7):
    t = train_tf(img)
    show = t.permute(1, 2, 0) * torch.tensor(STD) + torch.tensor(MEAN)   # 정규화 되돌리기
    ax[i].imshow(show.clamp(0, 1)); ax[i].set_title(f"증강 {i}"); ax[i].axis("off")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★★**: **같은 한 장인데 6번 모두 다릅니다.**
> 모델 입장에서는 **매 epoch 마다 새로운 이미지**를 보는 셈입니다.
> 그래서 **외우기가 어려워지고**, 그게 과적합을 줄입니다.

> **함정 ★**: `test_tf` 에는 `RandomCrop`·`Flip` 이 **없습니다.** 위 셀에서 직접 확인하세요.
> 매년 학생들이 `train_tf` 를 테스트에도 그대로 쓰고 *"정확도가 이상하다"* 고 합니다.

> `v2.ToImage()` + `v2.ToDtype(..., scale=True)` 조합이 예전 `ToTensor()` 를 대체합니다.

## 실습 8 — 증강 적용 후 재학습

In [ ]:
# 셀 3 — 2교시 CNN 을 그대로, 데이터만 증강
class CNN(nn.Module):                      # 2교시와 동일 (노트북에서는 재정의)
    def __init__(self, num_classes=10):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1),
                                            nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128))
        self.c = nn.Sequential(nn.Flatten(), nn.Dropout(0.3),
                               nn.Linear(128*4*4, 256), nn.ReLU(), nn.Linear(256, num_classes))
    def forward(self, x): return self.c(self.f(x))

def make_loaders(train_transform):
    full = datasets.CIFAR10("data", train=True, transform=train_transform)
    tr, _ = random_split(full, [45000, 5000], generator=torch.Generator().manual_seed(0))
    val_full = datasets.CIFAR10("data", train=True, transform=test_tf)     # ★ 검증은 증강 없이
    _, va = random_split(val_full, [45000, 5000], generator=torch.Generator().manual_seed(0))
    return (DataLoader(tr, batch_size=128, shuffle=True),
            DataLoader(va, batch_size=256))

loss_fn = nn.CrossEntropyLoss()

def evaluate(m, loader):
    m.eval(); c = t = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            c += (m(xb).argmax(1) == yb).sum().item(); t += yb.size(0)
    return c / t

def train_cnn(train_transform, epochs=10, tag=""):
    torch.manual_seed(0)
    model = CNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    tl, vl_loader = make_loaders(train_transform)
    accs = []
    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad(); loss.backward(); opt.step()
        accs.append(evaluate(model, vl_loader))
    print(f"{tag:10s} 최종 검증 정확도 {accs[-1]*100:5.2f}%")
    return model, accs

_, acc_plain = train_cnn(test_tf,  epochs=10, tag="증강 없음")
_, acc_aug   = train_cnn(train_tf, epochs=10, tag="증강 있음")

plt.plot([a*100 for a in acc_plain], marker="o", label="증강 없음")
plt.plot([a*100 for a in acc_aug],   marker="o", label="증강 있음")
plt.xlabel("epoch"); plt.ylabel("검증 정확도 (%)"); plt.legend()
plt.title("데이터 증강의 효과"); plt.show()

> **관찰 포인트 ★**: 초반에는 **증강 있는 쪽이 오히려 낮습니다** — 문제가 어려워졌으니까요.
> 그런데 **후반에 역전**되거나, 증강 없는 쪽이 먼저 정체합니다.
> 6주차에 드롭아웃에서 본 것과 **똑같은 모양**입니다 — *"훈련을 어렵게 해서 실전을 낫게 한다"*.

> ⚠️ 10 epoch 로는 역전이 안 보일 수도 있습니다. **증강의 효과는 긴 학습에서** 확실해집니다.
> 시간이 있으면 `epochs=20` 으로 다시 돌려 보세요.

## AMP — fp16 을 섞어 쓴다

```
   fp32 (32비트 실수)    4 byte    지금까지 쓴 것
   fp16 (16비트 실수)    2 byte    ★ 메모리 절반, 연산은 더 빠름

   AMP = Automatic Mixed Precision
        "곱셈같이 안전한 연산은 fp16 으로, 누적같이 민감한 것은 fp32 로"
        → PyTorch 가 알아서 섞어 준다
```

```
   fp16 이 표현할 수 있는 가장 작은 값 ≈ 6e-8

   역전파 중 기울기가 1e-9 이 되면?   →  fp16 에서 0 이 된다  ★ 언더플로
        → 그 파라미터는 학습이 멈춘다

   해법: 손실에 큰 수(예: 65536)를 곱해서 역전파하고,
        갱신 직전에 다시 나눈다   →  이걸 자동으로 하는 것이 GradScaler
```

> **핵심 메시지 ★★ (출제 지점)**: `GradScaler` 가 해결하는 것은 **기울기 언더플로**입니다.
> 빼면 fp16 에서 **기울기가 0 으로 죽어** 학습이 안 됩니다.

> 3주차에 *"GPU 는 빠르지만 좁다"* 고 했죠. **AMP 는 그 좁음을 푸는 첫 번째 기법**입니다.
> **9주차 ViT 미세조정, 14주차 확산 모델**에서 그대로 씁니다.

## 실습 9 — AMP 적용·측정 ★

In [ ]:
# 셀 4 — AMP 적용 (추가되는 코드는 세 줄뿐)
from torch.amp import autocast, GradScaler

def train_amp(use_amp, epochs=5):
    torch.manual_seed(0)
    model = CNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    tl, vl = make_loaders(train_tf)
    scaler = GradScaler(device, enabled=use_amp)          # ★ 추가 ①

    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for ep in range(epochs):
        model.train()
        for xb, yb in tl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            with autocast(device, enabled=use_amp):        # ★ 추가 ② 순전파만 감싼다
                loss = loss_fn(model(xb), yb)
            scaler.scale(loss).backward()                  # ★ 추가 ③ 손실을 키워서 역전파
            scaler.step(opt)
            scaler.update()

    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated()/1024**3 if device == "cuda" else 0
    acc = evaluate(model, vl)
    print(f"AMP={str(use_amp):5s} | 시간 {dt:6.1f}초 | 최대 VRAM {peak:4.2f}GB | 검증 {acc*100:5.2f}%")
    return dt, peak

t_fp32, m_fp32 = train_amp(False)
t_amp,  m_amp  = train_amp(True)

if device == "cuda":
    print(f"\n→ 속도 {t_fp32/t_amp:.2f}배, 메모리 {m_fp32/max(m_amp,1e-9):.2f}배 개선")
else:
    print("\n(CPU 에서는 AMP 이득이 없습니다. 결과 데이터로 해석만 하세요.)")

> **관찰 포인트 ★**: 눈여겨볼 것은 **VRAM** 입니다. 속도 이득이 작아도 **메모리는 확실히 줍니다.**
> 그것만으로도 **배치를 키우거나 더 큰 모델을 올릴 수 있습니다** — 9주차에 그렇게 씁니다.
> 개선폭은 **GPU 세대에 따라 크게 다릅니다.** 구형 GPU 에서는 속도 이득이 거의 없을 수도 있습니다.

> **함정 ★**: `autocast` 는 **순전파와 손실 계산만** 감쌉니다.
> `backward()` 를 그 안에 넣지 마세요. `scaler.scale(loss).backward()` 는 **밖**입니다.

In [ ]:
# 셀 5 — GradScaler 없이 fp16 을 돌리면 (30초 시연)
torch.manual_seed(0)
model = CNN().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
tl, _ = make_loaders(train_tf)

model.train()
for i, (xb, yb) in enumerate(tl):
    xb, yb = xb.to(device), yb.to(device)
    opt.zero_grad()
    with autocast(device, enabled=(device == "cuda")):
        loss = loss_fn(model(xb), yb)
    loss.backward()                                  # ★ scaler 없이 그냥
    g = model.f[0][0].weight.grad
    print(f"iter {i} | loss {loss.item():.4f} | 첫 Conv 기울기 |g| 평균 {g.abs().mean():.3e} "
          f"| 0 인 비율 {(g == 0).float().mean()*100:.1f}%")
    opt.step()
    if i == 4: break

> **결과 해석 ★★**: **기울기 중 0 인 비율**을 보세요. `GradScaler` 없이 fp16 을 쓰면
> 작은 기울기들이 **언더플로로 0 이 됩니다.** 그 파라미터는 학습이 멈춥니다.
> `scaler.scale(loss)` 가 손실을 키워서 이걸 막습니다.
> (CPU 에서는 `autocast` 가 꺼져 있어 이 현상이 안 나타납니다 — 정상입니다.)

| | fp32 | AMP(fp16) |
|---|---|---|
| 코드 | 지금까지 그대로 | **세 줄 추가** |
| 메모리 | 기준 | 대략 절반 |
| 속도 | 기준 | 1.5~3배 (GPU 세대에 따라) |
| 정확도 | 기준 | **거의 같다** ★ |

In [ ]:
# 셀 6 — 과제 제출용 표를 지금 만들어 둔다
print(f"{'설정':12s} {'학습시간(초)':>12s} {'최대 VRAM(GB)':>14s}")
print("-" * 40)
print(f"{'fp32':12s} {t_fp32:>12.1f} {m_fp32:>14.2f}")
print(f"{'AMP(fp16)':12s} {t_amp:>12.1f} {m_amp:>14.2f}")

---

### 과제 (마감 10/29 목 23:59 — **중간고사 다음 주**)

```
  ① 12_cnn_cifar10.ipynb (출력 저장)
  ② MLP vs CNN 비교표 (파라미터 수 · 정확도)
  ③ 증강 전후 검증 정확도
  ④ AMP 적용 전후 학습 시간 · VRAM   ← 위 셀 6 의 표
  ⑤ 회고 3줄
```

> 시험 준비와 겹치지 않게 마감을 일부러 뒤로 미뤘습니다. **시험부터 준비**하세요.

### 중간고사 (10/23 금 1교시, 50분 · 범위 1~7주)

> **출제 1순위 다섯 개**
> 1. **학습 루프 5단계의 순서**와 `zero_grad()` 가 필요한 이유 (4주차)
> 2. **활성함수가 없으면 층을 쌓아도 선형** (5주차)
> 3. **과적합의 진단법** — 훈련↓ + 검증↑ (6주차)
> 4. **MLP 가 이미지에 불리한 이유 두 가지** (7주차)
> 5. **출력 크기 계산** `(입력−커널+2×패딩)/스트라이드 + 1` (7주차)

### ★★ 9주차 준비물 — 지금 시작하세요

```
  ○ 클래스 3~5개  (예: 고양이/강아지/토끼,  또는 본인 관심 주제)
  ○ 클래스당 30~60장  (총 100~300장이면 충분)
  ○ 폴더 구조 : mydata/train/<클래스명>/*.jpg,  mydata/val/<클래스명>/*.jpg
  ○ 휴대폰 사진, 웹 수집 모두 가능 — 초상권·저작권 주의
```

> **8주차는 시험만 보고 끝날 수 있어서 오늘 미리 알립니다.**
> 준비 못 한 학생용 대체 데이터셋도 제공하지만, **본인 데이터로 하는 편이 훨씬 재미있습니다.**

### 이 노트북 체크리스트

- [ ] 같은 이미지의 증강본이 매번 다른 것을 봤다
- [ ] **검증셋에 증강을 적용하지 않는 이유**를 말할 수 있다 ★
- [ ] 증강 유무 검증 정확도를 비교했다
- [ ] AMP 를 적용해 **시간과 VRAM 을 측정**했다 ★
- [ ] `autocast` 가 감싸는 범위를 안다
- [ ] **`GradScaler` 가 필요한 이유**(기울기 언더플로)를 안다 ★★
- [ ] 중간고사 일시·범위·형식을 안다
- [ ] **9주차 준비물**을 알고 있다 ★★